In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def train_logistic_regression(X, y, lr=0.1, epochs=100):
    """
    對應論文公式 (5)：訓練邏輯回歸作為投影函數 (Base Learner)
    """
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    b = 0
    for _ in range(epochs):
        model = np.dot(X, w) + b
        predictions = sigmoid(model)
        # 梯度下降
        dw = (1 / n_samples) * np.dot(X.T, (predictions - y))
        db = (1 / n_samples) * np.sum(predictions - y)
        w -= lr * dw
        b -= lr * db
    return w, b

def simple_k_means_k2(X, max_iter=10):
    """
    對應步驟 2：k-means (k=2) 分配偽標籤 (Pseudo-label Assignment)
    """
    # 隨機初始化兩個中心點
    centroids = X[np.random.choice(X.shape[0], 2, replace=False)]
    for _ in range(max_iter):
        # 計算到中心點的距離並分配類別 (0 或 1)
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)
        # 更新中心點
        new_centroids = np.array([X[labels == k].mean(axis=0) for k in range(2)])
        if np.all(centroids == new_centroids): break
        centroids = new_centroids
    return labels

def erl_framework(target_data, source_data, N=10, r=5):
    """
    Ensemble Representation Learning (ERL) 主框架
    N: 集成個數 (Ensemble size)
    r: 每個原型集的樣本數 (Samples per prototype set)
    """
    n_target = target_data.shape[0]
    ensemble_w = []
    ensemble_b = []

    # 步驟 1 & 2 & 3: Ensemble Generation & Clustering & Learning
    for n in range(N):
        # 隨機選擇 r 個樣本作為原型集
        idx = np.random.choice(n_target, r, replace=False)
        prototypes = target_data[idx]

        # K-means 產生偽標籤
        pseudo_labels = simple_k_means_k2(prototypes)

        # 訓練投影函數 (Logistic Regression)
        w, b = train_logistic_regression(prototypes, pseudo_labels)
        ensemble_w.append(w)
        ensemble_b.append(b)

    # 步驟 4: Representation Construction (串接所有投影函數的輸出)
    def project(data):
        projections = []
        for w, b in zip(ensemble_w, ensemble_b):
            # 論文提到輸出為對應的類別預測值 (或概率)
            p = sigmoid(np.dot(data, w) + b)
            projections.append(p)
        # 沿著特徵維度串接 (Concatenating Outputs)
        return np.column_stack(projections)

    learned_target = project(target_data)
    learned_source = project(source_data)

    return learned_target, learned_source

# --- 模擬測試 ---
# 假設 z_t 是經由 Tangent Space Mapping 後的 100 個樣本，每個樣本 3 個特徵
z_t = np.random.randn(100, 3)
z_s = np.random.randn(50, 3)

new_z_t, new_z_s = erl_framework(z_t, z_s, N=5, r=10)

print(f"原始特徵維度: {z_t.shape[1]}")
print(f"ERL 學習後的特徵維度: {new_z_t.shape[1]} (等於集成個數 N)")
print(f"前 3 個樣本的學習表示:\n{new_z_t[:3]}")

原始特徵維度: 3
ERL 學習後的特徵維度: 5 (等於集成個數 N)
前 3 個樣本的學習表示:
[[0.97279837 0.85020556 0.28115457 0.62009216 0.14255155]
 [0.09330041 0.66928773 0.91827433 0.05171598 0.32217981]
 [0.49026979 0.7249424  0.74298844 0.18204641 0.22183802]]
